In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

import shap

RANDOM_STATE = 42

In [ ]:
train_data = pd.read_csv('/kaggle/input/competitions/santander-customer-transaction-prediction/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/santander-customer-transaction-prediction/test.csv')

print(train_data.shape)
print(test_data.shape)

train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
train_data.isnull().sum().sum()

In [ ]:
sns.countplot(x='target', data=train_data)
plt.title("Target Distribution")
plt.show()

In [ ]:
train_data.drop(['ID_code', 'target'], axis=1).hist(figsize=(20,20), bins=30)
plt.show()

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(train_data.iloc[:,1:20].corr(), cmap='coolwarm')
plt.show()

In [ ]:
X = train_data.drop(['ID_code', 'target'], axis=1)
y = train_data['target']

X_test = test_data.drop(['ID_code'], axis=1)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')

print("Baseline accuracy score", scores.mean())

In [ ]:
'''rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)

scores = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')
print("Random Forest ROC-AUC:", scores.mean())'''

In [ ]:
lgb_model = lgb.LGBMClassifier(device='gpu', n_estimators=1000, learning_rate=0.01, max_depth=-1, random_state=RANDOM_STATE)

scores = cross_val_score(lgb_model, X, y, cv=cv, scoring='roc_auc')
print("LightGBM ROC-AUC:", scores.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'num_leaves': [31, 50],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [500, 1000]
}

grid = GridSearchCV(
    lgb.LGBMClassifier(random_state=RANDOM_STATE),
    params,
    cv=3,
    scoring='roc_auc',
    verbose=1
)

grid.fit(X, y)

print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
final_model = lgb.LGBMClassifier(**grid.best_params_)
final_model.fit(X, y)

In [ ]:
preds = final_model.predict_proba(X_test)[:,1]

submission = pd.DataFrame({
    'ID_code': test_data['ID_code'],
    'target': preds
})

submission.to_csv('submission.csv', index=False)

In [ ]:
sample_X = X.sample(5000, random_state=42)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(sample_X)

shap.summary_plot(shap_values, sample_X)